<a href="https://colab.research.google.com/github/ivanlapin0017-debug/nlp-internet-genres-bert/blob/main/%D0%96%D0%B0%D0%BD%D1%80%D0%BE%D0%B2%D0%B0%D1%8F_%D0%BA%D0%BB%D0%B0%D1%81%D1%81%D0%B8%D1%84%D0%B8%D0%BA%D0%B0%D1%86%D0%B8%D1%8F_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import importlib, subprocess, sys
pkg = 'transformers'
if importlib.util.find_spec(pkg) is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, 'accelerate'])
else:
    print('transformers already installed')

transformers already installed


In [2]:
import torch
from torch.utils.data import Dataset

class GenresDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_len=512):
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        target = self.targets[idx]
        enc = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'text': text,
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'targets': torch.tensor(target, dtype=torch.long)
        }

In [3]:
import numpy as np
import random
import torch
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.optim import AdamW
from dataclasses import dataclass

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

@dataclass
class TrainStats:
    train_loss: float
    train_acc: float
    val_loss: float
    val_acc: float

class BertClassifier:
    def __init__(self,
                 model_path,
                 tokenizer_path,
                 n_classes=8,
                 epochs=1,
                 model_save_path='/content/bert.pt',
                 batch_size=16,
                 lr=2e-5,
                 weight_decay=0.01,
                 warmup_ratio=0.0,
                 grad_accum_steps=1,
                 use_amp=True):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = BertForSequenceClassification.from_pretrained(model_path)
        self.model.config.num_labels = n_classes
        self.model.classifier = torch.nn.Linear(self.model.config.hidden_size, n_classes)
        self.model.to(self.device)
        self.tokenizer = BertTokenizer.from_pretrained(tokenizer_path)
        self.max_len = 512
        self.epochs = epochs
        self.model_save_path = model_save_path
        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.warmup_ratio = warmup_ratio
        self.grad_accum_steps = max(1, grad_accum_steps)
        self.use_amp = use_amp and torch.cuda.is_available()
        self.scaler = torch.cuda.amp.GradScaler(enabled=self.use_amp)
        self.class_weights = None

    def preparation(self, X_train, y_train, X_valid, y_valid):
        self.train_set = GenresDataset(X_train, y_train, self.tokenizer, self.max_len)
        self.valid_set = GenresDataset(X_valid, y_valid, self.tokenizer, self.max_len)
        self.train_loader = DataLoader(self.train_set, batch_size=self.batch_size, shuffle=True)
        self.valid_loader = DataLoader(self.valid_set, batch_size=self.batch_size, shuffle=False)
        # Class weights (optional if imbalance)
        try:
            from sklearn.utils.class_weight import compute_class_weight
            classes = np.unique(y_train)
            cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
            self.class_weights = torch.tensor(cw, dtype=torch.float, device=self.device)
            print('Class weights:', self.class_weights.cpu().numpy())
        except Exception as e:
            print('Class weight computation skipped:', e)
        self.loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights).to(self.device)
        self.optimizer = AdamW(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        total_steps = len(self.train_loader) * self.epochs // self.grad_accum_steps
        warmup_steps = int(total_steps * self.warmup_ratio)
        self.scheduler = get_linear_schedule_with_warmup(self.optimizer, warmup_steps, total_steps)

    def _run_batch(self, batch, train=True):
        input_ids = batch['input_ids'].to(self.device)
        attention_mask = batch['attention_mask'].to(self.device)
        targets = batch['targets'].to(self.device)
        with torch.cuda.amp.autocast(enabled=self.use_amp):
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            loss = self.loss_fn(outputs.logits, targets)
        preds = torch.argmax(outputs.logits, dim=1)
        if train:
            self.scaler.scale(loss).backward()
        return loss, preds, targets

    def fit_epoch(self):
        self.model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        self.optimizer.zero_grad()
        for step, batch in enumerate(self.train_loader):
            loss, preds, targets = self._run_batch(batch, train=True)
            total_loss += loss.item()
            correct += (preds == targets).sum().item()
            total += targets.size(0)
            if (step + 1) % self.grad_accum_steps == 0:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.scheduler.step()
                self.optimizer.zero_grad()
        return correct / total, total_loss / len(self.train_loader)

    def eval_epoch(self):
        self.model.eval()
        total_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in self.valid_loader:
                loss, preds, targets = self._run_batch(batch, train=False)
                total_loss += loss.item()
                correct += (preds == targets).sum().item()
                total += targets.size(0)
        return correct / total, total_loss / len(self.valid_loader)

    def train(self, early_stopping_patience=2):
        best_acc = 0.0
        patience = 0
        for epoch in range(1, self.epochs + 1):
            print(f'Epoch {epoch}/{self.epochs}')
            train_acc, train_loss = self.fit_epoch()
            val_acc, val_loss = self.eval_epoch()
            print(f'Train loss {train_loss:.4f} acc {train_acc:.4f} | Val loss {val_loss:.4f} acc {val_acc:.4f}')
            if val_acc > best_acc:
                torch.save(self.model.state_dict(), self.model_save_path)
                best_acc = val_acc
                patience = 0
                print('  (saved model)')
            else:
                patience += 1
                if patience >= early_stopping_patience:
                    print('Early stopping triggered')
                    break
        # load best
        self.model.load_state_dict(torch.load(self.model_save_path, map_location=self.device))
        self.model.eval()

    def predict(self, text: str):
        self.model.eval()
        enc = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        with torch.no_grad():
            input_ids = enc['input_ids'].to(self.device)
            attention_mask = enc['attention_mask'].to(self.device)
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            pred = torch.argmax(outputs.logits, dim=1).item()
        return pred

In [4]:
import pandas as pd
import csv
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
import chardet
from collections import Counter

file_path = 'CorpJokesVK1.csv'
with open(file_path, 'rb') as f:
    encoding = chardet.detect(f.read())['encoding']
print('Detected encoding:', encoding)

def try_read(path, encoding):
    for delim in [';', ',']:
        try:
            df_ = pd.read_csv(path, delimiter=delim, encoding=encoding)
            print(f'Loaded with delimiter {delim}')
            return df_
        except Exception:
            pass
    with open(path, 'r', encoding=encoding) as f:
        dialect = csv.Sniffer().sniff(f.readline(), delimiters=';,')
    df_ = pd.read_csv(path, delimiter=dialect.delimiter, encoding=encoding)
    print(f'Loaded with inferred delimiter {dialect.delimiter}')
    return df_

df = try_read(file_path, encoding)
df = df.dropna(subset=['text', 'category']).reset_index(drop=True)
le = preprocessing.LabelEncoder()
df['category'] = le.fit_transform(df['category'])
print('Label classes:', list(le.classes_))

# Stratified split train/val/test (70/10/20 example)
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['category'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.125, stratify=train_df['category'], random_state=42)  # 0.125 * 0.8 = 0.10 total
print('Sizes:', len(train_df), len(val_df), len(test_df))
print('Train dist:', Counter(train_df['category']))
print('Val   dist:', Counter(val_df['category']))
print('Test  dist:', Counter(test_df['category']))

train_data = train_df
valid_data = val_df
test_data  = test_df

FileNotFoundError: [Errno 2] No such file or directory: 'CorpJokesVK1.csv'

In [ ]:
df.head()

In [ ]:
torch.cuda.empty_cache()
classifier = BertClassifier(
    model_path='DeepPavlov/rubert-base-cased',
    tokenizer_path='DeepPavlov/rubert-base-cased',
    n_classes=len(le.classes_),
    epochs=2,
    batch_size=16,
    lr=2e-5,
    warmup_ratio=0.06,
    grad_accum_steps=1,
    model_save_path='/content/bert_state.pt'
)

In [ ]:
classifier.preparation(
    X_train=list(train_data['text']),
    y_train=list(train_data['category']),
    X_valid=list(valid_data['text']),
    y_valid=list(valid_data['category'])
)

In [ ]:
classifier.train()

In [ ]:
texts = list(test_data['text'])
labels = list(test_data['category'])
predictions = [classifier.predict(t) for t in texts]
from sklearn.metrics import precision_recall_fscore_support, classification_report
precision, recall, f1score = precision_recall_fscore_support(labels, predictions, average='macro')[:3]
print(f'precision: {precision:.4f}, recall: {recall:.4f}, f1score: {f1score:.4f}')
print(classification_report(labels, predictions))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(labels, predictions)  # fixed order (y_true, y_pred)
print(cm)
ConfusionMatrixDisplay(cm).plot()